In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../")

C:\Users\Lenovo\Desktop\Diabetes_Classifier


In [ ]:
from modeling.XGBoost import XGBoost
from modeling.Ada import Ada
from modeling.MLP import MLP
from modeling.RandomForest import RandomForest
from modeling.Ensemble import LR,StackingEnsemble
from src.configuration import CLEAN_DATA,MODEL_PATH
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

c:\Users\Lenovo\anaconda3\envs\Diabetes\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

df=pd.read_csv(CLEAN_DATA)
df.head()
df.columns = df.columns.str.strip()

In [6]:
xgb=XGBoost()
xgb_params={'n_estimators': 7800, 'max_depth': 13, 'learning_rate': 0.03585979413065538, 'gamma': 0.00025046646096832056, 'min_child_weight': 8, 'reg_alpha': 9.979812055651985, 'reg_lambda': 0.041477402642739976, 'subsample': 0.9999355054935987, 'colsample_bytree': 0.8896668801909641, 'tree_method': 'hist', 'n_jobs': -1}
xgb.set_params(xgb_params)

rf=RandomForest()
rf_params={'n_estimators': 1200, 'max_depth': 6, 'min_samples_split': 18, 'min_samples_leaf': 9, 'max_features': 'log2', 'bootstrap': True, 'criterion': 'gini', 'class_weight': None, 'n_jobs': -1, 'random_state': 42}
rf.set_params(rf_params)

ada=Ada()
ada_params={'n_estimators': 364, 'learning_rate': 0.6133741065916875}
ada.set_params(ada_params)

mlp=MLP()
mlp_params={'hidden_layer_sizes': (64,), 'activation': 'relu', 'solver': 'adam', 'alpha': 6.120954197759074e-05, 'learning_rate_init': 0.006507372434299954, 'batch_size': 64, 'max_iter': 900, 'early_stopping': True, 'random_state': 42}
mlp.set_params(mlp_params)


Setting the parameters for the Base models!

In [7]:
meta_x_model=LR()
params={'penalty': 'l2', 'l1_ratio': 0.377184927025366, 'C': 0.3926479245177274, 'class_weight': 'balanced','n_jobs':-1}
meta_x_model.set_params(params)

Setting the parameters for the Meta model!

In [ ]:
stack=StackingEnsemble([xgb,rf,mlp,ada],meta_x_model,df,5,'diagnosis',['XGBoost,RandomForest,Ada'])

Instantiating StackingEnsemble Object !

In [9]:
stack.fit_base_models()

Fits the base models!

In [10]:
stack.fit_meta_model()

XGBoost fold 1/5 done
XGBoost fold 2/5 done
XGBoost fold 3/5 done
XGBoost fold 4/5 done
XGBoost fold 5/5 done
RandomForest fold 1/5 done
RandomForest fold 2/5 done
RandomForest fold 3/5 done
RandomForest fold 4/5 done
RandomForest fold 5/5 done
MLP fold 1/5 done
MLP fold 2/5 done
MLP fold 3/5 done
MLP fold 4/5 done
MLP fold 5/5 done
Ada fold 1/5 done
Ada fold 2/5 done
Ada fold 3/5 done
Ada fold 4/5 done
Ada fold 5/5 done
      XGBoost_pred  RandomForest_pred  MLP_pred  Ada_pred
1215      0.416197           0.455491  0.385598  0.495663
19        0.088586           0.159017  0.043013  0.369981
2093      0.038265           0.040798  0.027734  0.377956
668       0.058976           0.053313  0.037806  0.407496
218       0.233063           0.253950  0.271230  0.461056
...            ...                ...       ...       ...
4426      0.896069           0.774983  0.884366  0.560844
466       0.192961           0.285293  0.216142  0.406356
3092      0.158395           0.297926  0.292274  0.46

Fits the Meta model with OOF from base models!

In [11]:
m=stack.cv(5)
print(m)
m.out()

ModelMetrics(accuracy=array([0.82947625, 0.81851401, 0.83069428, 0.82317073, 0.82560976]), balanced_accuracy=array([0.82644755, 0.8163523 , 0.83459767, 0.82427918, 0.81859052]), precision=array([0.76035503, 0.74344023, 0.74585635, 0.74220963, 0.76615385]), f1=array([0.78593272, 0.77389985, 0.79528719, 0.78325859, 0.77691108]), recall=array([0.81329114, 0.80696203, 0.85173502, 0.82911392, 0.78797468]), roc_auc=array([0.90979446, 0.90785813, 0.92508512, 0.91872614, 0.90713532]))
Accuracy: 0.8254930037729122, std: 0.004405279421601714
Precision: 0.7516030188590005, std: 0.009759254268063762
F1: 0.7830578856736159, std: 0.007476813377828245
Recall: 0.8178153575849538, std: 0.02147989765303116
Balanced Accuracy:0.8240534433689346, std: 0.0064205067142632565
ROC-AUC: 0.9137198348402276, std: 0.007033588806173316


Cross validating stack!

In [12]:
stack.measure()

{'accuracy': 0.8352826510721247, 'balanced': 0.8366712501931817, 'precision': 0.7666666666666667, 'f1': 0.8032596041909197, 'recall': 0.843520782396088, 'roc_auc': 0.9275340495258626}


In [13]:
input=pd.DataFrame({'age':[80],'gender':[1],'bmi':[25],'chol':[5.5],'tg':[5.8],'hdl':[4.8],'ldl':[4.2],'cr':[64],'bun':[4.8]})
result=stack.predict_single(input)
print(result)

You have diabetes!


Testing a single row - to see if its ready for deployment!

In [ ]:

stack.save(MODEL_PATH)

Stacking ensemble saved to models/StackEnsemble.pkl


Saving the Stacking Ensemble!